# 02 — CLI tour

Exercises every `sonar-probe` subcommand against synthetic data:

- `sonar-probe extract` (speech)
- `sonar-probe extract-text`
- `sonar-probe compare` (scalar, matrix long/wide, cross-modal)
- `sonar-probe inspect`

**Prerequisite.** The CLI loads real SONAR weights — no fake-encoder injection is exposed on the command line. Run this notebook on a machine where `pip install -e .[dev]` succeeded (Linux + CUDA recommended). The first speech or text extract will download the SONAR checkpoint; set `HF_HOME` to a persistent directory to cache.

In [ ]:
import os, tempfile, wave
from pathlib import Path
import numpy as np

WORK = Path(tempfile.mkdtemp(prefix="sonar-probe-cli-"))
os.environ["SONAR_PROBE_WORK"] = str(WORK)
print("workdir:", WORK)

## Synthetic audio corpus

In [ ]:
SR = 16000

def write_tone(path: Path, hz: float, seconds: float) -> None:
    t = np.arange(int(SR * seconds)) / SR
    x = (0.3 * np.sin(2 * np.pi * hz * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR); w.writeframes(x.tobytes())

audio = WORK / "audio"; audio.mkdir()
write_tone(audio / "rec01.wav", 440.0, 6.0)
write_tone(audio / "rec02.wav", 880.0, 6.0)

(WORK / "wav.scp").write_text(f"rec01 {audio/'rec01.wav'}\nrec02 {audio/'rec02.wav'}\n")
(WORK / "segments.tsv").write_text(
    "file_id\tchannel\tspkid\tstart\tend\tlang\n"
    "rec01\t0\tspk1\t0.0\t2.0\teng\n"
    "rec01\t0\tspk1\t2.0\t4.0\teng\n"
    "rec02\t0\tspk2\t0.0\t3.0\teng\n"
    "rec02\t0\tspk2\t3.0\t6.0\teng\n"
)
(WORK / "texts.tsv").write_text(
    "text_id\ttext\tlang\n"
    "q1\ta quick brown fox\teng\n"
    "q2\tthe lazy dog sleeps\teng\n"
    "q3\tun renard brun rapide\tfra\n"
)
sorted(p.name for p in WORK.iterdir())

## `extract` — plain_text source → Kaldi ark/scp sink

In [ ]:
!sonar-probe extract \
    --source plain_text \
    --input "$SONAR_PROBE_WORK/segments.tsv" \
    --wav-scp "$SONAR_PROBE_WORK/wav.scp" \
    --sink kaldi_ark \
    --output "$SONAR_PROBE_WORK/speech_store" \
    --speech-lang eng \
    --batch-size 4

In [ ]:
!ls "$SONAR_PROBE_WORK/speech_store"
!head -3 "$SONAR_PROBE_WORK/speech_store/metadata.jsonl"

## `extract-text` — TSV of texts → HuggingFace dataset sink

In [ ]:
!sonar-probe extract-text \
    --input "$SONAR_PROBE_WORK/texts.tsv" \
    --sink hf_dataset \
    --output "$SONAR_PROBE_WORK/text_store" \
    --batch-size 4

## `inspect` — summarize any store

In [ ]:
!sonar-probe inspect "$SONAR_PROBE_WORK/speech_store"
!sonar-probe inspect "$SONAR_PROBE_WORK/text_store"

## `compare` — scalar (1×1) result

Two store IDs → a single cosine printed to stdout.

In [ ]:
!sonar-probe compare \
    --a "store:$SONAR_PROBE_WORK/speech_store#rec01-0-spk1-0-2000" \
    --b "store:$SONAR_PROBE_WORK/speech_store#rec01-0-spk1-2000-4000"

## `compare` — cross-modal (speech ↔ text)

`text:<text>@<lang>` encodes a literal string on the fly.

In [ ]:
!sonar-probe compare \
    --a "store:$SONAR_PROBE_WORK/speech_store#rec01-0-spk1-0-2000" \
    --b "text:a tone at four hundred forty hertz@eng"

## `compare` — matrix output (long TSV, filtered)

A full store on each side yields an `(N, M)` matrix. `--output-format long` writes `(a_id, b_id, score)` TSV rows; `--top-k` keeps the best-K per row; `--threshold` drops low scores.

In [ ]:
!sonar-probe compare \
    --a "$SONAR_PROBE_WORK/speech_store" \
    --b "$SONAR_PROBE_WORK/text_store" \
    --output-format long \
    --top-k 2 \
    --output "$SONAR_PROBE_WORK/matches.tsv"
!head "$SONAR_PROBE_WORK/matches.tsv"

## `compare` — matrix output (wide CSV)

`--output-format wide` emits a full `(N+1, M+1)` CSV matrix. Row/column headers are the operand IDs; `--top-k` / `--threshold` are ignored in wide mode (matrix shape is preserved).

In [ ]:
!sonar-probe compare \
    --a "$SONAR_PROBE_WORK/speech_store" \
    --b "$SONAR_PROBE_WORK/text_store" \
    --output-format wide \
    --output "$SONAR_PROBE_WORK/matrix.csv"
!cat "$SONAR_PROBE_WORK/matrix.csv"

## Append & strict modes

- `--append` adds to an existing `hf_dataset` store (the `kaldi_ark` sink currently overwrites — see README's known limitations).
- `--strict` turns any soft error (bad segment, decode failure, encode failure) fatal. Without `--strict`, soft errors are logged to `<output>/failures.jsonl` and the run continues.